# 06 — MEM Electricity | Gold KPIs

Electricity KPIs from January 2020 (last six years plus the current year), by distributor (Edenorte, Edesur, Edeeste) and total EDE.

| KPI | Table · column | Definition |
|---|---|---|
| ENE-001 Electricity tariff (applied) | `electricity_tariff_monthly.applied_dop` | Regulated tariff charged (RD$), by tariff (BTS1…MTH), concept and distributor |
| ENE-002 Reference vs applied tariff gap | `electricity_tariff_monthly.tariff_gap_pct` | (Reference / applied − 1) × 100; a proxy of the tariff subsidy |
| ENE-003 Typical residential bill | `electricity_bts1_bill.applied_bill_dop` | BTS-1 bill for 200 and 300 kWh/month: fixed charge (101+ kWh) + energy blocks |
| ENE-004 Typical bill YoY % | `electricity_bts1_bill.applied_bill_yoy_pct` | vs same month last year |
| ENE-005 Average selling price per kWh | `electricity_metrics.selling_price_dop_kwh` | Published by MEM, RD$ per kWh billed |
| ENE-006 Selling price change | `selling_price_mom_pct`, `selling_price_yoy_pct` | vs previous month / same month last year |
| ENE-007 Purchase price and margin | `purchase_price_cusd_kwh`, `margin_cusd_kwh` | US cents per kWh bought; selling minus purchase price |
| ENE-008 Energy losses | `losses_pct`, `losses_rolling_pct` | Month and 12-month rolling, % of energy bought |
| ENE-009 Collection and CRI | `collection_rolling_pct`, `cri_rolling_pct` | 12-month rolling, % (CRI = cash recovery index) |

Tariff periods are expanded to months; when two published periods overlap (April–May 2026) the later column in the file is used. Before October 2021 the applied tariff is one value for all distributors.
**Status:** Development.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
S_EDE, S_TAR = f"{CATALOG}.silver.mem_ede_monthly", f"{CATALOG}.silver.mem_tariffs_quarterly"
G_M, G_T, G_B = f"{CATALOG}.gold.electricity_metrics", f"{CATALOG}.gold.electricity_tariff_monthly", f"{CATALOG}.gold.electricity_bts1_bill"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
START = "2020-01-01"
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)
r4 = lambda c: F.round(c, 4).cast("decimal(14,4)")

# ---------- Distributor metrics ----------
pick = {("Precio Medio de Venta de Energía", "DOP/kWh"): "sell_dop", ("Precio Medio de Venta de Energía", "cUSD/kWh"): "sell_cusd",
        ("Precio Medio de Compra de Energía", "cUSD/kWh"): "buy_cusd", ("Energía Facturada", "GWh"): "billed_gwh",
        ("Compra de Energía", "GWh"): "bought_gwh", ("Pérdidas", "%"): "loss", ("Pérdidas - Año Movil", "%"): "loss_r",
        ("Cobranzas - Año Movil", "%"): "coll_r", ("CRI - Año Movil", "%"): "cri_r"}
e = spark.table(S_EDE)
cond = F.lit(None)
for (ind, unit), alias in pick.items():
    cond = F.when((F.col("indicator") == ind) & (F.col("unit") == unit), F.lit(alias)).otherwise(cond)
p = (e.withColumn("k", cond).filter("k IS NOT NULL").groupBy("reference_month", "distributor").pivot("k", list(pick.values())).agg(F.first("value")))
w = Window.partitionBy("distributor").orderBy("reference_month")
m = (p.withColumn("prev", F.lag("sell_dop").over(w)).withColumn("prev_m", F.lag("reference_month").over(w))
      .withColumn("ly", F.lag("sell_dop", 12).over(w)).withColumn("ly_m", F.lag("reference_month", 12).over(w)))
gm = m.filter(F.col("reference_month") >= START).select(
    "reference_month", "distributor",
    r4(F.col("sell_dop")).alias("selling_price_dop_kwh"), r4(F.col("sell_cusd")).alias("selling_price_cusd_kwh"),
    r4(F.when(F.months_between("reference_month", "prev_m") == 1, (F.col("sell_dop") / F.col("prev") - 1) * 100)).alias("selling_price_mom_pct"),
    r4(F.when(F.months_between("reference_month", "ly_m") == 12, (F.col("sell_dop") / F.col("ly") - 1) * 100)).alias("selling_price_yoy_pct"),
    r4(F.col("buy_cusd")).alias("purchase_price_cusd_kwh"), r4(F.col("sell_cusd") - F.col("buy_cusd")).alias("margin_cusd_kwh"),
    r4(F.col("bought_gwh")).alias("energy_bought_gwh"), r4(F.col("billed_gwh")).alias("energy_billed_gwh"),
    r4(F.col("loss")).alias("losses_pct"), r4(F.col("loss_r")).alias("losses_rolling_pct"),
    r4(F.col("coll_r")).alias("collection_rolling_pct"), r4(F.col("cri_r")).alias("cri_rolling_pct"),
    F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"))

# ---------- Tariffs by month ----------
t = spark.table(S_TAR)
months = t.select("period_order", F.explode(F.sequence("period_start", "period_end", F.expr("interval 1 month"))).alias("reference_month")).distinct()
tm = t.join(months, "period_order")
dists = spark.createDataFrame([("Edenorte",), ("Edesur",), ("Edeeste",)], "d string")
tm = (tm.join(dists, (tm.distributor == dists.d) | (tm.distributor == "Todas")).withColumn("distributor", F.col("d")).drop("d"))
wl = Window.partitionBy("reference_month", "tariff", "concept", "tariff_type", "distributor").orderBy(F.col("period_order").desc())
tm = tm.withColumn("rn", F.row_number().over(wl)).filter("rn = 1").drop("rn")
gt = (tm.groupBy("reference_month", "distributor", "tariff", "concept")
        .agg(F.max(F.when(F.col("tariff_type") == "reference", F.col("value_dop"))).alias("reference_dop"),
             F.max(F.when(F.col("tariff_type") == "applied", F.col("value_dop"))).alias("applied_dop"),
             F.max(F.when(F.col("tariff_type") == "applied", F.col("period_label"))).alias("period_label"))
        .withColumn("tariff_gap_pct", r4((F.col("reference_dop") / F.col("applied_dop") - 1) * 100))
        .withColumn("reference_dop", r4(F.col("reference_dop"))).withColumn("applied_dop", r4(F.col("applied_dop")))
        .withColumn("_gold_run_id", F.lit(RUN_ID)).withColumn("_gold_processed_ts", F.lit(RUN_TS).cast("timestamp")))

# ---------- Typical BTS-1 bill ----------
FIX = "Cargo Fijo por Rango de Consumo — (ii) Consumo mensual de 101 kwh en adelante"
B1, B2 = "Cargo por Energía — (i) Los primeros kwh entre 0 y 200", "Cargo por Energía — (ii) Los siguientes kwh entre 201 y 300"
bts = gt.filter((F.col("tariff") == "BTS1") & F.col("concept").isin(FIX, B1, B2))
pv = lambda col: bts.groupBy("reference_month", "distributor").pivot("concept", [FIX, B1, B2]).agg(F.first(col))
a, r = pv("applied_dop"), pv("reference_dop")
bill = lambda df, kwh: df[f"`{FIX}`"] + F.least(F.lit(kwh), F.lit(200)) * df[f"`{B1}`"] + F.greatest(F.lit(kwh - 200), F.lit(0)) * df[f"`{B2}`"]
rows = []
for kwh in (200, 300):
    rows.append(a.select("reference_month", "distributor", F.lit(kwh).alias("monthly_kwh"), bill(a, kwh).alias("applied_bill_dop"))
                 .join(r.select("reference_month", "distributor", bill(r, kwh).alias("reference_bill_dop")), ["reference_month", "distributor"], "left"))
gb = rows[0].unionByName(rows[1])
wb = Window.partitionBy("distributor", "monthly_kwh").orderBy("reference_month")
gb = (gb.withColumn("ly", F.lag("applied_bill_dop", 12).over(wb)).withColumn("ly_m", F.lag("reference_month", 12).over(wb))
        .select("reference_month", "distributor", "monthly_kwh", r4(F.col("applied_bill_dop")).alias("applied_bill_dop"),
                r4(F.col("reference_bill_dop")).alias("reference_bill_dop"),
                r4((F.col("reference_bill_dop") / F.col("applied_bill_dop") - 1) * 100).alias("bill_gap_pct"),
                r4(F.when(F.months_between("reference_month", "ly_m") == 12, (F.col("applied_bill_dop") / F.col("ly") - 1) * 100)).alias("applied_bill_yoy_pct"),
                F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts")))

In [0]:
checks = []
def add(table, name, rtype, tested, failed):
    checks.append((RUN_ID, table, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
n1, n2, n3 = gm.count(), gt.count(), gb.count()
add(G_M, "unique_month_distributor", "enforced", n1, n1 - gm.select("reference_month", "distributor").distinct().count())
add(G_M, "selling_price_present", "enforced", n1, gm.filter("selling_price_dop_kwh IS NULL").count())
add(G_T, "unique_month_distributor_tariff_concept", "enforced", n2, n2 - gt.select("reference_month", "distributor", "tariff", "concept").distinct().count())
add(G_T, "applied_tariff_present", "observational", n2, gt.filter("applied_dop IS NULL").count())
add(G_B, "bill_complete", "enforced", n3, gb.filter("applied_bill_dop IS NULL").count())
# Reconciliation with the MEM PDF report (May 2026): Edenorte 10.13, Edesur 10.51 RD$/kWh
rec = {r["distributor"]: float(r["selling_price_dop_kwh"]) for r in gm.filter("reference_month = '2026-05-01'").collect()}
add(G_M, "reconciles_mem_pdf_may_2026", "enforced", 2, sum(1 for d, v in (("Edenorte", 10.13), ("Edesur", 10.51)) if abs(rec.get(d, 0) - v) > 0.005))
for c in checks: print(f"  {c[6]:<4} {c[1].split('.')[-1]:<28} {c[2]:<42} failed={c[5]:,}/{c[4]:,}")
assert not [c for c in checks if c[6] == "FAIL"]

for df, tbl, order, cmt in [(gm, G_M, ["distributor", "reference_month"], "Gold: electricity distributor KPIs ENE-005 to ENE-009 (MEM), month x distributor, from 2020."),
                            (gt, G_T, ["tariff", "concept", "distributor", "reference_month"], "Gold: regulated tariffs by month (ENE-001) and reference vs applied gap (ENE-002), RD$, from 2020."),
                            (gb, G_B, ["monthly_kwh", "distributor", "reference_month"], "Gold: typical BTS-1 residential bill for 200 and 300 kWh (ENE-003, ENE-004), RD$/month, from 2020.")]:
    df.orderBy(*order).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl)
    spark.sql(f"COMMENT ON TABLE {tbl} IS '{cmt}'")
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

last = spark.sql(f"""SELECT distributor, selling_price_dop_kwh, selling_price_yoy_pct, purchase_price_cusd_kwh, losses_rolling_pct, cri_rolling_pct
  FROM {G_M} WHERE reference_month = (SELECT MAX(reference_month) FROM {G_M}) ORDER BY distributor""")
display(last)
lb = spark.sql(f"SELECT reference_month, distributor, monthly_kwh, applied_bill_dop, reference_bill_dop FROM {G_B} WHERE reference_month IN ('2020-01-01', (SELECT MAX(reference_month) FROM {G_B})) AND monthly_kwh = 200 ORDER BY 1, 2")
rng = spark.sql(f"SELECT MIN(reference_month) a, MAX(reference_month) z FROM {G_M}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "metrics_rows": n1, "tariff_rows": n2, "bill_rows": n3, "months": [str(rng["a"]), str(rng["z"])],
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks},
    "latest": [{k: str(v) for k, v in x.asDict().items()} for x in last.collect()],
    "bill_200": [{k: str(v) for k, v in x.asDict().items()} for x in lb.collect()]}, ensure_ascii=False))